# Chat local avec un endpoint Databricks Llama

Notebook **local** (Jupyter classique, pas de notebook Databricks).

Contrairement à la version précédente (agent LangGraph + outils Unity Catalog + Vector Search Retriever encapsulés dans un `ResponsesAgent` MLflow déployé en Model Serving), ce notebook fait simplement :

1. une connexion au workspace Databricks via le SDK
2. un appel chat direct (sans outils, sans RAG, sans graphe) à un endpoint de serving Llama
3. une boucle de conversation qui garde l'historique en mémoire

## Prérequis

```bash
pip install databricks-sdk
```

Variables d'environnement à définir (ou via un profil `~/.databrickscfg`) :

```bash
export DATABRICKS_HOST="https://<votre-workspace>.cloud.databricks.com"
export DATABRICKS_TOKEN="dapi..."
```

In [ ]:
from databricks.sdk import WorkspaceClient

# Nom de l'endpoint de serving Llama (Foundation Model API ou endpoint custom)
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-70b-instruct"

# Prompt système optionnel
SYSTEM_PROMPT = "Tu es un assistant utile et concis."

# Nombre max de messages d'historique conservés (pour éviter de dépasser le contexte)
MAX_HISTORY_MESSAGES = 20

# Le client lit DATABRICKS_HOST / DATABRICKS_TOKEN automatiquement,
# ou utilise le profil par défaut de ~/.databrickscfg
w = WorkspaceClient()

## Fonction d'appel au endpoint

Pas de LangGraph, pas de `ResponsesAgent`, pas d'outils : un simple appel chat-completion.

In [ ]:
def call_llama(messages: list[dict]) -> str:
    """
    Appelle l'endpoint de serving Llama avec une liste de messages
    au format chat-completion ({'role': ..., 'content': ...}) et
    retourne le texte de la réponse.
    """
    response = w.serving_endpoints.query(
        name=LLM_ENDPOINT_NAME,
        messages=messages,
    )
    return response.choices[0].message.content

## Boucle de chat

L'historique de conversation est gardé en mémoire dans une simple liste Python (au lieu de l'`AgentState` LangGraph).

In [ ]:
history: list[dict] = []
if SYSTEM_PROMPT:
    history.append({"role": "system", "content": SYSTEM_PROMPT})


def chat(user_message: str) -> str:
    """Envoie un message utilisateur, met à jour l'historique, retourne la réponse."""
    history.append({"role": "user", "content": user_message})

    # Tronque l'historique si besoin (garde le system prompt + les derniers échanges)
    trimmed = history
    if len(history) > MAX_HISTORY_MESSAGES:
        system_msgs = [m for m in history if m["role"] == "system"]
        other_msgs = [m for m in history if m["role"] != "system"]
        trimmed = system_msgs + other_msgs[-MAX_HISTORY_MESSAGES:]

    answer = call_llama(trimmed)
    history.append({"role": "assistant", "content": answer})
    return answer

## Exemple d'utilisation ponctuelle

In [ ]:
print(chat("Bonjour, peux-tu te présenter en une phrase ?"))

## Boucle interactive (optionnelle)

Exécutez cette cellule pour discuter en continu depuis le notebook local. Tapez `exit` pour quitter.

In [ ]:
while True:
    user_input = input("Vous: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    reply = chat(user_input)
    print(f"Assistant: {reply}\n")